# NERVE — train your own model (Colab)

Trains a [NERVE](https://huggingface.co/Phips/NERVE) model on your own data.

**How to use**
1. Put your images in Google Drive (or upload them to the Colab session).
2. Run the cells below in order.
3. In the last cell: pick **scale** and **dataset type**, fill in your folder
   paths, and press **Start training**.

> The folder fields are pre-filled with the **tiny sample dataset bundled with
> traiNNer-redux**, so you can press **Start training** straight away to check
> everything works (set iterations to ~2000 for a 1-minute test). Replace them
> with your own folders for real training.

- **paired** = you already have matching HR and LR folders.
- **otf** = you only have HR; the LR is degraded on the fly (Real-ESRGAN style)
  — this is what the released real-world NERVE models use.

Checkpoints and a TensorBoard log land in `experiments/<name>/` in the Colab
session (copy them to Drive if you want to keep them).


In [ ]:
#@title 1. Check the GPU
!nvidia-smi -L || echo "No GPU found - go to Runtime > Change runtime type and pick a GPU"


In [ ]:
#@title 2. (Optional) Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
#@title 3. Install traiNNer-redux + the NERVE architecture
import os, subprocess, sys

REPO = "https://github.com/the-database/traiNNer-redux.git"
ARCH_URL = "https://huggingface.co/Phips/NERVE/resolve/main/nerve_arch.py"
UTIL_URL = "https://huggingface.co/Phips/NERVE/resolve/main/icnr.py"

if not os.path.isdir("traiNNer-redux"):
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)

# the NERVE arch + its ICNR helper
subprocess.run(["curl", "-sL", ARCH_URL, "-o", "traiNNer-redux/traiNNer/archs/nerve_arch.py"], check=True)
subprocess.run(["curl", "-sL", UTIL_URL, "-o", "traiNNer-redux/traiNNer/utils/icnr.py"], check=True)

os.chdir("traiNNer-redux")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("Installed. Working dir:", os.getcwd())


In [ ]:
"""Config builder shared by the NERVE Colab notebook.

Kept as a normal, testable module (build_colab_notebook.py embeds it), so the
generated configs can be validated locally before shipping.
"""

from __future__ import annotations

from pathlib import Path

_HEAD = """\
name: {name}
scale: {scale}
use_amp: true
amp_bf16: true
use_channels_last: true
use_compile: true
compile_mode: default
num_gpu: auto
manual_seed: 1024
"""

_PAIRED = """\

datasets:
  train:
    name: Train
    type: pairedimagedataset
    dataroot_gt: [{hr}]
    dataroot_lq: [{lr}]
    lq_size: 96
    use_hflip: true
    use_rot: true
    num_worker_per_gpu: 2
    batch_size_per_gpu: 8
"""

_OTF = """\

high_order_degradation: true
lq_usm: true
lq_usm_radius_range: [1, 25]
blur_prob: 0.6
resize_prob: [0.2, 0.7, 0.1]
resize_mode_list: ['bilinear', 'bicubic', 'nearest-exact', 'lanczos']
resize_mode_prob: [0.25, 0.25, 0.25, 0.25]
resize_range: [0.4, 1.5]
gaussian_noise_prob: 0.4
noise_range: [1, 15]
jpeg_prob: 0.4
jpeg_range: [40, 95]
resize_mode_list3: ['bilinear', 'bicubic', 'nearest-exact', 'lanczos']
resize_mode_prob3: [0.25, 0.25, 0.25, 0.25]

datasets:
  train:
    name: Train
    type: realesrgandataset
    dataroot_gt: [{hr}]
    blur_kernel_size: 12
    kernel_list: ['iso', 'aniso', 'generalized_iso']
    kernel_prob: [0.45, 0.35, 0.2]
    kernel_range: [5, 17]
    blur_sigma: [0.2, 2]
    lq_size: 96
    use_hflip: true
    use_rot: true
    num_worker_per_gpu: 2
    batch_size_per_gpu: 8
"""

_VAL = """\
  val:
    name: Val
    type: pairedimagedataset
    dataroot_gt: [{vhr}]
    dataroot_lq: [{vlr}]
"""

_TRAIN = """\

network_g:
  type: nerve

{path_block}
train:
  ema_decay: 0.999
  ema_power: 0.75
  grad_clip: false
  optim_g:
    type: AdamW
    lr: !!float 2e-4
    weight_decay: 0
    betas: [0.9, 0.99]
  scheduler:
    type: MultiStepLR
    milestones: [{m1}, {m2}]
    gamma: 0.5
  total_iter: {iters}
  losses:
{losses}
val:
  val_enabled: {val_enabled}
  val_freq: 5000
  save_img: false
  tile_size: 0
  tile_overlap: 8
  metrics_enabled: true
  metrics:
    psnr:
      type: calculate_psnr
      crop_border: {scale}
      test_y_channel: true
    ssim:
      type: calculate_ssim
      crop_border: {scale}
      test_y_channel: true

logger:
  print_freq: 100
  save_checkpoint_freq: 5000
  save_checkpoint_format: safetensors
  use_tb_logger: true
"""

_BASE_LOSS = "    - type: charbonnierloss\n      loss_weight: 1.0\n"
_OTF_LOSS = _BASE_LOSS + "    - type: mssimloss\n      loss_weight: 0.5\n"


def build_config(
    *,
    scale: int,
    mode: str,
    hr: str,
    lr: str = "",
    val_hr: str = "",
    val_lr: str = "",
    iters: int = 500000,
    name: str = "my_NERVE",
    pretrain: str = "",
) -> tuple[str, str]:
    """Return (run_name, yaml_text). mode is 'paired' or 'otf'."""
    if mode not in ("paired", "otf"):
        raise ValueError(f"mode must be 'paired' or 'otf', got {mode!r}")
    if not hr:
        raise ValueError("HR folder is required")
    if mode == "paired" and not lr:
        raise ValueError("LR folder is required for paired mode")

    run_name = f"{scale}x_{name}"
    cfg = _HEAD.format(name=run_name, scale=scale)
    cfg += (_PAIRED if mode == "paired" else _OTF).format(hr=hr, lr=lr)

    has_val = bool(val_hr and val_lr)
    if has_val:
        cfg += _VAL.format(vhr=val_hr, vlr=val_lr)

    if pretrain:
        path_block = (
            "path:\n"
            f"  pretrain_network_g: {pretrain}\n"
            "  strict_load_g: false  # warm-start: mismatched keys (e.g. a different scale) are skipped\n"
            "  resume_state: ~\n"
        )
    else:
        path_block = "path:\n  param_key_g: ~\n  strict_load_g: true\n  resume_state: ~\n"

    cfg += _TRAIN.format(
        path_block=path_block,
        m1=int(iters * 0.4),
        m2=int(iters * 0.7),
        iters=iters,
        losses=_BASE_LOSS if mode == "paired" else _OTF_LOSS,
        val_enabled="true" if has_val else "false",
        scale=scale,
    )
    return run_name, cfg


def write_config(run_name: str, yaml_text: str, base: str | Path = ".") -> Path:
    out = Path(base) / "options" / "train" / "NERVE" / f"{run_name}.yml"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(yaml_text)
    return out


# ---- simple GUI ----
import os
import subprocess
import sys

import ipywidgets as widgets
from IPython.display import display

PRETRAINS = {
    "None (from scratch)": None,
    "2x release (clean)": "models/2x_NERVE_release.safetensors",
    "4x release (clean)": "models/4x_NERVE_release.safetensors",
    "4x OTF fidelity": "models/4x_NERVE_OTF_fidelity.safetensors",
    "4x OTF GAN": "models/4x_NERVE_OTF_gan.safetensors",
    "2x OTF GAN": "models/2x_NERVE_OTF_gan.safetensors",
}
scale_w = widgets.Dropdown(options=[("4x", 4), ("2x", 2), ("1x", 1)], value=4, description="Scale")
pre_w = widgets.Dropdown(options=list(PRETRAINS), value="None (from scratch)", description="Warm-start")
mode_w = widgets.Dropdown(
    options=[("OTF degradation (HR only)", "otf"), ("Paired HR/LR", "paired")],
    value="otf", description="Data")
hr_w = widgets.Text(value="datasets/train/dataset1/hr", description="HR folder")
lr_w = widgets.Text(value="datasets/train/dataset1/lr", description="LR folder")
vhr_w = widgets.Text(description="Val HR", placeholder="(optional)")
vlr_w = widgets.Text(description="Val LR", placeholder="(optional)")
it_w = widgets.IntText(value=500000, description="Iterations")
name_w = widgets.Text(value="my_NERVE", description="Name")
go = widgets.Button(description="Start training", button_style="success")
out = widgets.Output()

display(scale_w, mode_w, pre_w, hr_w, lr_w, vhr_w, vlr_w, it_w, name_w, go, out)


def on_start(_):
    out.clear_output()
    with out:
        pretrain_path = ""
        rel = PRETRAINS[pre_w.value]
        if rel:
            from huggingface_hub import hf_hub_download

            print(f"Downloading warm-start model: {rel}")
            pretrain_path = hf_hub_download("Phips/NERVE", rel)
            print("  ->", pretrain_path)

        try:
            run_name, cfg = build_config(
                scale=scale_w.value, mode=mode_w.value, hr=hr_w.value,
                lr=lr_w.value, val_hr=vhr_w.value, val_lr=vlr_w.value,
                iters=it_w.value, name=name_w.value, pretrain=pretrain_path)
        except ValueError as e:
            print("Please check the form:", e)
            return
        path = write_config(run_name, cfg)
        print(f"Config written to {path}\n")
        print(cfg)
        subprocess.run([sys.executable, "train.py", "-opt", str(path), "--auto_resume"], check=False)


go.on_click(on_start)
